In [1]:
import cadquery as cq
from cadquery import exporters
import math
from jupyter_cadquery import *

from jupyter_cadquery.replay import replay, enable_replay, disable_replay

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

versions()

Overwriting auto display for cadquery Workplane and Shape

Versions:
- jupyter_cadquery  4.0.2
- cad_viewer_widget 3.0.2
- open cascade      7.7.2.1



In [2]:
cv = open_viewer("Brace", cad_width=800, height=600)
set_default_viewer("Brace")

In [3]:
# Post parameters
post_width = 4.0       # Y
post_depth = 6.0       # X
post_height = 96.0     # Z

# Sill parameters
sill_width = 8.0       # Y
sill_height = 10.0      # Z
sill_length = 72.0     # X

# Brace parameters
brace_thickness = 4.0  # Y
brace_depth = 6.0      # perpendicular to brace axis in local Z
brace_run = 40.0       # X distance
brace_rise = 60.0      # Z distance

# Tenon parameters
tenon_length = 3.0
tenon_thickness = 1.5
tenon_overlap = 0.10

# Mortise parameters
clearance = 0.02
mortise_extra_depth = 0.25
mortise_overlap = 0.05

# Sill datum
outside_face_y = sill_width / 2
post_center_y = outside_face_y - post_width / 2
brace_center_y = outside_face_y - brace_thickness / 2

In [4]:

# Brace angle degrees
brace_angle = math.degrees(
    math.atan2(brace_rise, brace_run)
)

# Brace angle radians
a = math.radians(brace_angle)

# Brace length
brace_length = math.hypot(
    brace_run,
    brace_rise,
)

# Z-dimensions
z_bottom = -brace_depth / 2
z_top = brace_depth / 2

# Upper shoulder
upper_x_bottom = -z_bottom * math.tan(a)
upper_x_top = -z_top * math.tan(a)

upper_shoulder_run = abs(
    upper_x_bottom - upper_x_top
)

# Lower shoulder
lower_x_bottom = (
    brace_length
    + z_bottom / math.tan(a)
)

lower_x_top = (
    brace_length
    + z_top / math.tan(a)
)

lower_shoulder_run = abs(
    lower_x_top - lower_x_bottom
)

print(f"brace angle: {brace_angle:.2f}°")
print(f"brace length: {brace_length:.3f}\"")
print(f"lower shoulder run: {lower_shoulder_run:.3f}\"")
print(f"upper shoulder run: {upper_shoulder_run:.3f}\"")

brace angle: 56.31°
brace length: 72.111"
lower shoulder run: 4.000"
upper shoulder run: 9.000"


In [5]:
brace_body_local = (
    cq.Workplane("XZ")
    .moveTo(upper_x_bottom, z_bottom)
    .lineTo(upper_x_top, z_top)
    .lineTo(lower_x_top, z_top)
    .lineTo(lower_x_bottom, z_bottom)
    .close()
    .extrude(
        brace_thickness / 2,
        both=True,
    )
)

In [6]:
def brace_local_prism(x0, x1, y0, y1, z0, z1):
    return cq.Solid.makeBox(
        x1 - x0,
        y1 - y0,
        z1 - z0,
        cq.Vector(x0, y0, z0),
    )

In [7]:
def place_on_brace(shape):
    return (
        shape
        .rotate(
            (0, 0, 0),
            (0, 1, 0),
            brace_angle,
        )
        .translate(
            (
                0,
                brace_center_y,
                brace_rise,
            )
        )
    )

In [8]:
post = cq.Solid.makeBox(
    post_depth,
    post_width,
    post_height,
    cq.Vector(
        -post_depth,
        outside_face_y - post_width,
        0,
    ),
)

In [9]:
sill = cq.Solid.makeBox(
    sill_length,
    sill_width,
    sill_height,
    cq.Vector(
        -12,
        -sill_width / 2,
        -sill_height,
    ),
)

In [10]:
brace_body = place_on_brace(
    brace_body_local
).val()

In [11]:
#Tenon position
a = math.radians(brace_angle)

tenon_y0 = -tenon_thickness / 2
tenon_y1 =  tenon_thickness / 2

# Tenon is flush with this face/arris of the brace.
tenon_z0 = -brace_depth / 2
tenon_z1 = tenon_z0 + brace_depth

# Mortise position
mortise_y0 = tenon_y0 - clearance / 2
mortise_y1 = tenon_y1 + clearance / 2

mortise_z0 = tenon_z0 - clearance / 2
mortise_z1 = tenon_z1 + clearance / 2

In [12]:
brace_z_bottom = -brace_depth / 2

# Bottom corner of post-end angled cut
upper_shoulder_bottom_x = (
    -brace_z_bottom * math.tan(a)
)

# Bottom corner of sill-end angled cut
lower_shoulder_bottom_x = (
    brace_length
    + brace_z_bottom / math.tan(a)
)

In [13]:
# Upper tenon
upper_tenon_root_x = upper_x_bottom
upper_tenon_length = upper_shoulder_run

upper_tenon_local = brace_local_prism(
    upper_tenon_root_x - upper_tenon_length,
    upper_tenon_root_x + tenon_overlap,

    tenon_y0,
    tenon_y1,

    tenon_z0,
    tenon_z1,
)

upper_tenon = place_on_brace(upper_tenon_local)

In [14]:
# Lower tenon
lower_tenon_root_x = lower_x_bottom
lower_tenon_length = lower_shoulder_run

lower_tenon_local = brace_local_prism(
    lower_tenon_root_x - tenon_overlap,
    lower_tenon_root_x + lower_tenon_length,

    tenon_y0,
    tenon_y1,

    tenon_z0,
    tenon_z1,
)

lower_tenon = place_on_brace(lower_tenon_local)

In [15]:
brace = (
    brace_body
    .fuse(upper_tenon)
    .fuse(lower_tenon)
    .clean()
)

In [16]:
# Upper mortise
upper_mortise_local = brace_local_prism(
    upper_tenon_root_x
        - upper_tenon_length
        - mortise_extra_depth,

    upper_tenon_root_x
        + mortise_overlap,

    mortise_y0,
    mortise_y1,

    mortise_z0,
    mortise_z1,
)

upper_mortise = place_on_brace(
    upper_mortise_local
)

In [17]:
# Lower mortise
lower_mortise_local = brace_local_prism(
    lower_tenon_root_x
        - mortise_overlap,

    lower_tenon_root_x
        + lower_tenon_length
        + mortise_extra_depth,

    mortise_y0,
    mortise_y1,

    mortise_z0,
    mortise_z1,
)

lower_mortise = place_on_brace(
    lower_mortise_local
)

In [18]:
post_cut = post.cut(upper_mortise).clean()
sill_cut = sill.cut(lower_mortise).clean()

In [19]:
show_object(post_cut, name="post_cut")
show_object(sill_cut, name="sill_cut")
show_object(brace, name="brace")

+
++
+++
